In [ ]:
import pandas as pd
import pickle

# import re
from langchain_community.cache import SQLiteCache
from langchain.globals import set_llm_cache

set_llm_cache(SQLiteCache(database_path=".langchain.db"))
file_name = "merge-company_risk_data_with_embedding"
# file_name = "250528-company_risk_data_with_embedding"
snapshot_file_path = f"result/{file_name}-10percent_edges.pkl"
snapshot_data_list = pickle.load(open(snapshot_file_path, "rb"))
real_data_path = f"result/{file_name}.pkl"
raw_risk_data = pickle.load(open(real_data_path, "rb"))
company = "PCG"
print(len(snapshot_data_list))
snapshot_data_list[:3]

198


[{'data': {'id': 'risk_0',
   'label': 'Accounting errors',
   'raw_size': np.float64(5404.95210390441),
   'size_level': 1,
   'size': 1,
   'color': 'rgb(54, 162, 235)',
   'risk_level': 1,
   'story': ''},
  'position': {'x': 121.99445071219743, 'y': 579.2076987690455}},
 {'data': {'id': 'risk_1',
   'label': 'Business interruption from fire hazards',
   'raw_size': np.float64(5923.85113281553),
   'size_level': 2,
   'size': 50,
   'color': 'rgb(54, 162, 235)',
   'risk_level': 2,
   'story': ''},
  'position': {'x': 223.39578668030333, 'y': 189.48241312851215}},
 {'data': {'id': 'risk_2',
   'label': 'Business interruption from flood',
   'raw_size': np.float64(5814.38038305227),
   'size_level': 1,
   'size': 50,
   'color': 'rgb(54, 162, 235)',
   'risk_level': 1,
   'story': ''},
  'position': {'x': 634.4006060142402, 'y': 530.7914594506092}}]

compare 4o-mini, 4.1-mini, o3-mini.

In [2]:
from collections import Counter


def get_risk_data_from_risk_name(risk_name, risk_data_list):
    for risk in risk_data_list:
        if risk["risk"] == risk_name:
            return risk
    return None


interested_risk_name_list = [
    "Intense market competition",
    "New competitor into the market",
    "Product substitute",
    "Uncompetitive service",
]
raw_risk_selected_keys = [
    "risk",
    "risk_desc",
    "rootcause",
]
raw_risk_selected_data_list = []
selected_raw_risk_data = [i for i in raw_risk_data if i["company"] == company]
for i in selected_raw_risk_data:
    raw_risk_selected_data_list.append({k: i[k] for k in raw_risk_selected_keys})
node_data_list = []
edge_data_list = []
for node_edge in snapshot_data_list:
    if node_edge["data"].get("source") is not None:
        # it a edge
        # edge_company = node_edge["data"]["source"].split("_")[1]
        # if edge_company != "PCG":
        #     continue
        source_risk_name = node_edge["data"]["source_risk_data"]["risk"]
        target_risk_name = node_edge["data"]["target_risk_data"]["risk"]
        if (
            source_risk_name in interested_risk_name_list
            or target_risk_name in interested_risk_name_list
        ):
            # print(source_risk_name, target_risk_name)
            edge_data_list.append(node_edge["data"])
    else:
        node_data_list.append(node_edge["data"])
print(len(edge_data_list))

7


In [3]:
edge_data_list

[{'source': 'risk_24',
  'target': 'risk_32',
  'weight': 4,
  'raw_weight': np.float64(126.22446226597216),
  'color': 'rgb(169, 169, 169)',
  'arrow_weight': 'none',
  'do_not_cal_weight': False,
  'edge_relation_reason': '',
  'source_risk_data': {'risk': 'Intense market competition',
   'risk_desc': 'A highly competitive market environment where businesses within the same industry are fiercely competing for customers and market share, potentially leading to decreased profitability and market share. ตลาดอาหารสัตว์เลี้ยงมีอัตราการเติบโตต่อเนื่องทุกปี  ทำให้เป็นแรงจูงใจให้ผู้ผลิตและผู้จัดจำหน่ายสินค้ากลุ่มอื่นๆ ทั้งรายเก่ามีการขยายธุรกิจมากขึ้น และรายใหม่ๆเข้ามาในธุรกิจนี้มากขึ้นด้วย,ความเสี่ยงจากการแข่งขันในตลาดบนช่องทางออนไลน์และอีคอมเมิร์ซ การแข่งขันที่รุนแรงจากแบรนด์เดิมและคู่แข่งรายใหม่ อาจส่งผลต่อยอดขายและการรักษาฐานลูกค้า,สถานการณ์ที่ธุรกิจต่างๆในอุตสาหกรรมเดียวกันมีการแข่งขันที่รุนแรง ซึ่งอาจส่งผลกระทบต่อกำไรและส่วนแบ่งตลาด,ขาดกิจกรรมที่ตอบสนองต่อความต้องการของลูกค้า,มีคู่แข่ง

In [4]:
# raise

In [5]:
from test_dynamic_pydantic import create_dynamic_pydantic_model
import pydantic
from typing import Type, Literal
import json

# old
# def create_edge_relationship_model(
#     risk_a: str, risk_b: str
# ) -> Type[pydantic.BaseModel]:
#     """
#     Creates an EdgeRelationship model with specific risk names.

#     Args:
#         risk_a (str): Name of the first risk
#         risk_b (str): Name of the second risk

#     Returns:
#         Type[pydantic.BaseModel]: A Pydantic model class for the edge relationship
#     """
#     relationship_choices_dict = {
#         f"{risk_a} lead to {risk_b}": f"{risk_a} is the cause and {risk_b} is the effect. Changes in {risk_a} directly influence or trigger {risk_b}.",
#         f"{risk_b} lead to {risk_a}": f"{risk_b} is the cause and {risk_a} is the effect. Changes in {risk_b} directly influence or trigger {risk_a}.",
#         "be a cause to each other": f"Both risks influence each other in a bidirectional relationship. Changes in either risk can affect the other.",
#         "correlated": f"The risks tend to occur together or show similar patterns, but there may not be a direct causal relationship between them.",
#         "no relationship": f"There is no significant connection or influence between these risks. They operate independently of each other.",
#     }
#     schema_definition_dict = {
#         "model_name": "EdgeRelationship",
#         "fields": {
#             "classification": (
#                 Literal[
#                     f"{risk_a} lead to {risk_b}",
#                     f"{risk_b} lead to {risk_a}",
#                     "be a cause to each other",
#                     "correlated",
#                     "no relationship",
#                 ],
#                 pydantic.Field(
#                     description=f"Select the type of relationship between {risk_a} and {risk_b}. Available choices: {json.dumps(relationship_choices_dict)}",
#                 ),
#             ),
#             "reason": (
#                 str,
#                 pydantic.Field(
#                     description="Brief reason for the relationship classification in 1-3 sentences"
#                 ),
#             ),
#         },
#     }

#     return create_dynamic_pydantic_model(
#         schema_definition_dict["model_name"],
#         schema_definition_dict["fields"],
#     )


# old#2
# def create_edge_relationship_model(
#     risk_a: str, risk_b: str
# ) -> Type[pydantic.BaseModel]:
#     """
#     Creates an EdgeRelationship model with specific risk names.

#     Args:
#         risk_a (str): Name of the first risk
#         risk_b (str): Name of the second risk

#     Returns:
#         Type[pydantic.BaseModel]: A Pydantic model class for the edge relationship
#     """
#     relationship_choices_dict = {
#         f"cause_effect": f"{risk_a} is the cause and {risk_b} is the effect. Changes in {risk_a} directly influence or trigger {risk_b}.",
#         f"effect_cause": f"{risk_b} is the cause and {risk_a} is the effect. Changes in {risk_b} directly influence or trigger {risk_a}.",
#         "bidirectional_relationship": f"Both risks influence each other in a bidirectional relationship. Changes in either risk can affect the other.",
#         "correlated": f"The risks tend to occur together or show similar patterns, but there may not be a direct causal relationship between them.",
#         "no_relationship": f"There is no significant connection or influence between these risks. They operate independently of each other.",
#     }
#     schema_definition_dict = {
#         "model_name": "EdgeRelationship",
#         "fields": {
#             "classification": (
#                 Literal[
#                     "cause_effect",
#                     "effect_cause",
#                     "bidirectional_relationship",
#                     "correlated",
#                     "no_relationship",
#                 ],
#                 pydantic.Field(
#                     description=f"Select the type of relationship between {risk_a} and {risk_b}. Available choices: {json.dumps(relationship_choices_dict)}",
#                 ),
#             ),
#             "reason": (
#                 str,
#                 pydantic.Field(
#                     description="Brief reason for the relationship classification in 1-3 sentences"
#                 ),
#             ),
#         },
#     }

#     return create_dynamic_pydantic_model(
#         schema_definition_dict["model_name"],
#         schema_definition_dict["fields"],
#     )


def create_edge_relationship_model(
    risk_a: str, risk_b: str
) -> Type[pydantic.BaseModel]:
    """
    Creates an EdgeRelationship model with specific risk names.

    Args:
        risk_a (str): Name of the first risk
        risk_b (str): Name of the second risk

    Returns:
        Type[pydantic.BaseModel]: A Pydantic model class for the edge relationship
    """

    schema_definition_dict = {
        "model_name": "EdgeRelationship",
        "fields": {
            f"{risk_a} lead to {risk_b}": (
                bool,
                pydantic.Field(
                    description=f"{risk_a} is the cause and {risk_b} is the effect.",
                ),
            ),
            f"{risk_a} lead to {risk_b} reason": (
                str,
                pydantic.Field(
                    description="Brief reason for the relationship classification in 1-3 sentences",
                ),
            ),
            f"{risk_b} lead to {risk_a}": (
                bool,
                pydantic.Field(
                    description=f"{risk_b} is the cause and {risk_a} is the effect.",
                ),
            ),
            f"{risk_b} lead to {risk_a} reason": (
                str,
                pydantic.Field(
                    description="Brief reason for the relationship classification in 1-3 sentences",
                ),
            ),
        },
    }

    return create_dynamic_pydantic_model(
        schema_definition_dict["model_name"],
        schema_definition_dict["fields"],
    )


sample_model = create_edge_relationship_model(
    risk_a="Intense market competition", risk_b="New competitor into the market"
)
sample_model.model_json_schema()

{'properties': {'Intense market competition lead to New competitor into the market': {'description': 'Intense market competition is the cause and New competitor into the market is the effect.',
   'title': 'Intense Market Competition Lead To New Competitor Into The Market',
   'type': 'boolean'},
  'Intense market competition lead to New competitor into the market reason': {'description': 'Brief reason for the relationship classification in 1-3 sentences',
   'title': 'Intense Market Competition Lead To New Competitor Into The Market Reason',
   'type': 'string'},
  'New competitor into the market lead to Intense market competition': {'description': 'New competitor into the market is the cause and Intense market competition is the effect.',
   'title': 'New Competitor Into The Market Lead To Intense Market Competition',
   'type': 'boolean'},
  'New competitor into the market lead to Intense market competition reason': {'description': 'Brief reason for the relationship classification i

In [6]:
from typing import Any, Dict, Optional
from langchain_openai import ChatOpenAI


def get_llm(provider: str = "openai", model_name: Optional[str] = None):
    """Initialize an LLM based on the provider."""
    if provider == "openai":
        model = model_name if model_name else "gpt-4o-mini"
        temperature = None if model_name in ["o3-mini"] else 0.1
        return ChatOpenAI(model=model, temperature=temperature)

    else:
        raise ValueError(f"Unsupported provider: {provider}")


# test the model
# llm_4omini = get_llm("openai")
# llm_4omini.invoke("Hello, how are you?")
llm_41mini = get_llm("openai", "gpt-4.1-mini")
llm_41mini.invoke("Hello, how are you?")
# llm_o3mini = get_llm("openai", "o3-mini")
# llm_o3mini.invoke("Hello, how are u?")

AIMessage(content="Hello! I'm doing great, thank you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 13, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6f2eabb9a5', 'id': 'chatcmpl-BjNChymGDsFQ9rnlmY76WSHuAfcWT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--6caf937c-adc7-42f6-9c4e-9d5fbf9eec89-0', usage_metadata={'input_tokens': 13, 'output_tokens': 16, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from langchain.prompts.chat import ChatPromptTemplate
from typing import Any, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser


def classify_edge_relationship(
    model_name: str, risk_a: Dict[str, Any], risk_b: Dict[str, Any]
) -> dict:
    try:

        llm = get_llm(model_name=model_name)
        EdgeRelationship = create_edge_relationship_model(
            risk_a=risk_a["risk"], risk_b=risk_b["risk"]
        )
        parser = PydanticOutputParser(pydantic_object=EdgeRelationship)

        # Format input data for the prompt
        risk_a_str = "\n".join([f"- {k}: {v}" for k, v in risk_a.items()])
        risk_b_str = "\n".join([f"- {k}: {v}" for k, v in risk_b.items()])
        risk_a_name = risk_a["risk"]
        risk_b_name = risk_b["risk"]
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    f"""You are an expert in risk assessment and business analysis. Your task is to classify the relationship between two risks based on their descriptions and context. your analysis will be from the perspective of business owner.

Analyze the provided data for {risk_a_name} and {risk_b_name} and determine the relationship type.



Provide the classification and a brief reason in the specified pydantic format.

""",
                ),
                (
                    "human",
                    f"Please classify the relationship between {risk_a_str} and  {risk_b_str}.",
                ),
            ]
        )
        structured_llm = llm.with_structured_output(EdgeRelationship)
        chain = prompt | structured_llm

        # Invoke the chain
        result = chain.invoke({})
        # print(f"{risk_a_str=}")
        # Return the relationship field from the parsed model
        return result

    except Exception as e:
        print(f"Error classifying edge relationship: {str(e)}")
        raise Exception(f"Error classifying edge relationship: {str(e)}")

In [8]:
len(edge_data_list), company

(7, 'PCG')

In [18]:
node_data_df = pd.DataFrame(node_data_list)
print(node_data_df.columns)
relation_counter = Counter()
relation_result = {}
all_result_list = []
for model_name in ["o3-mini", "gpt-4o-mini", "gpt-4.1-mini"]:
    for edge_data in edge_data_list:
        source_risk_id = edge_data["source"]
        # node_company = ("_").join(source_risk_id.split("_")[1:-1])
        # print(f"{node_company=}")
        # if node_company != company:
        #     continue
        target_risk_id = edge_data["target"]
        source_risk_name = node_data_df[node_data_df["id"] == source_risk_id][
            "label"
        ].values[0]
        target_risk_name = node_data_df[node_data_df["id"] == target_risk_id][
            "label"
        ].values[0]
        print(f"\t{source_risk_name=}")
        print(f"\t{target_risk_name=}")

        risk_a = get_risk_data_from_risk_name(
            source_risk_name, raw_risk_selected_data_list
        )
        risk_b = get_risk_data_from_risk_name(
            target_risk_name, raw_risk_selected_data_list
        )
        print("here")
        print(risk_a, risk_b)
        edge_relationship_a_b = classify_edge_relationship(
            model_name=model_name, risk_a=risk_a, risk_b=risk_b
        )
        edge_relationship_b_a = classify_edge_relationship(
            model_name=model_name, risk_a=risk_b, risk_b=risk_a
        )

        # all_result_list.append(
        #     {
        #         "model_name": model_name,
        #         "source_risk_name": source_risk_name,
        #         "target_risk_name": target_risk_name,
        #         "edge_relationship_a_b_classification": edge_relationship_a_b.model_dump(
        #             mode="json"
        #         )[
        #             "classification"
        #         ],
        #         "edge_relationship_a_b_reason": edge_relationship_a_b.model_dump(
        #             mode="json"
        #         )["reason"],
        #         "edge_relationship_b_a_classification": edge_relationship_b_a.model_dump(
        #             mode="json"
        #         )[
        #             "classification"
        #         ],
        #         "edge_relationship_b_a_reason": edge_relationship_b_a.model_dump(
        #             mode="json"
        #         )["reason"],
        #     }
        # )
        def postprocess_edge_relationship(
            edge_relationship_dict, source_risk_name, target_risk_name, prefix
        ):
            new_dict = {}
            for k, v in edge_relationship_dict.items():
                if k.find(source_risk_name) == 0:
                    if k.find("reason") != -1:
                        new_k = "cause_effect_reason"
                    else:
                        new_k = "cause_effect"
                elif k.find(target_risk_name) == 0:
                    if k.find("reason") != -1:
                        new_k = "effect_cause_reason"
                    else:
                        new_k = "effect_cause"
                else:
                    raise
                new_dict[prefix + new_k] = v
            return new_dict

        edge_relationship_a_b_dict = edge_relationship_a_b.model_dump(mode="json")
        final_relationship_a_b = postprocess_edge_relationship(
            edge_relationship_a_b_dict, source_risk_name, target_risk_name, "a_b_"
        )
        edge_relationship_b_a_dict = edge_relationship_b_a.model_dump(mode="json")
        final_relationship_b_a = postprocess_edge_relationship(
            edge_relationship_b_a_dict, target_risk_name, source_risk_name, "b_a_"
        )
        all_result_list.append(
            {
                "model_name": model_name,
                "source_risk_name": source_risk_name,
                "target_risk_name": target_risk_name,
                **final_relationship_a_b,
                **final_relationship_b_a,
            }
        )

# pickle.dump(new_edge_data_dict, open(edge_relationship_path, "wb"))

Index(['id', 'label', 'raw_size', 'size_level', 'size', 'color', 'risk_level',
       'story'],
      dtype='object')
	source_risk_name='Intense market competition'
	target_risk_name='New competitor into the market'
here
{'risk': 'Intense market competition', 'risk_desc': 'ตลาดอาหารสัตว์เลี้ยงมีอัตราการเติบโตต่อเนื่องทุกปี  ทำให้เป็นแรงจูงใจให้ผู้ผลิตและผู้จัดจำหน่ายสินค้ากลุ่มอื่นๆ ทั้งรายเก่ามีการขยายธุรกิจมากขึ้น และรายใหม่ๆเข้ามาในธุรกิจนี้มากขึ้นด้วย,ตลาดอาหารสุนัขมีการเติบโตอย่างต่อเนื่อง มีสินค้าใหม่และ Brand ใหม่เกิดขึ้นในตลาดจำนวนมาก และหลายแบรนด์หลายบริษัทมีการทำการตลาดเชิงรุก หรืออาจจะมีการใช้กลยุทธการขายสินค้าราคาถูกมากในกลุ่มสินค้าที่ price sensitive เพื่อแย่งลูกค้าและส่วนแบ่งทางการตลาด ส่งผลให้ทาง PCG อาจจะไม่สามารถทำยอดขาย อัตราการเติบโต รวมถึงส่วนแบ่งตลาด ได้ตามที่วางแผนไว้,มีคู่แข่งรายใหม่เข้าสู่ตลาดหรือคู่แข่งเดิมใช้กลยุทธ์การตลาดเชิงรุก ส่งผลให้ยอดขายไม่เป็นไปตามเป้าหมาย,ความเสี่ยงจากการแข่งขันในตลาดบนช่องทางออนไลน์และอีคอมเมิร์ซ การแข่งขันที่รุนแรงจากแบรนด์เดิมและคู

In [19]:
# create dataframe from all_result_list and save to exel
import pandas as pd
import pickle


df = pd.DataFrame(all_result_list)
df.to_excel("compare_new_relation_model.xlsx", index=False)

In [17]:
df.shape

(21, 7)

In [ ]:
'ตลาดอาหารสัตว์เลี้ยงมีอัตราการเติบโตต่อเนื่องทุกปี  ทำให้เป็นแรงจูงใจให้ผู้ผลิตและผู้จัดจำหน่ายสินค้ากลุ่มอื่นๆ ทั้งรายเก่ามีการขยายธุรกิจมากขึ้น และรายใหม่ๆเข้ามาในธุรกิจนี้มากขึ้นด้วย,ตลาดอาหารสุนัขมีการเติบโตอย่างต่อเนื่อง มีสินค้าใหม่และ Brand ใหม่เกิดขึ้นในตลาดจำนวนมาก และหลายแบรนด์หลายบริษัทมีการทำการตลาดเชิงรุก หรืออาจจะมีการใช้กลยุทธการขายสินค้าราคาถูกมากในกลุ่มสินค้าที่ price sensitive เพื่อแย่งลูกค้าและส่วนแบ่งทางการตลาด ส่งผลให้ทาง PCG อาจจะไม่สามารถทำยอดขาย อัตราการเติบโต รวมถึงส่วนแบ่งตลาด ได้ตามที่วางแผนไว้,มีคู่แข่งรายใหม่เข้าสู่ตลาดหรือคู่แข่งเดิมใช้กลยุทธ์การตลาดเชิงรุก ส่งผลให้ยอดขายไม่เป็นไปตามเป้าหมาย,ความเสี่ยงจากการแข่งขันในตลาดบนช่องทางออนไลน์และอีคอมเมิร์ซ การแข่งขันที่รุนแรงจากแบรนด์เดิมและคู่แข่งรายใหม่ อาจส่งผลต่อยอดขายและการรักษาฐานลูกค้า,ขาดกิจกรรมที่ตอบสนองต่อความต้องการของลูกค้า,สถานการณ์ที่ธุรกิจต่างๆในอุตสาหกรรมเดียวกันมีการแข่งขันที่รุนแรง ซึ่งอาจส่งผลกระทบต่อกำไรและส่วนแบ่งตลาด', 'rootcause': 'rootcause :High number of competitors in the market: จำนวนคู่แข่งในตลาดมีจำนวนมากทั้งในส่วนของจำนวน Manufacturer และจำนวน Brand , Aggressive pricing strategies: หลาย Brand ในตลาดกลุ่ม Economy และ Super Economy ใช้กลยุทธการแข่งขันด้านราคาเป็นหลัก ซึ่งทาง PCG อาจจะไม่สามารถแข่งขันโดยการใช้กลุยุทธการทำราคาถูกมากได้ , Other: คู่แข่งมีการทำตลาดอย่างเข้มข้นมากขึ้นทั้งส่วนของการใช้ Media การทำโปรโมชั่น และกิจกรรมต่างๆ ที่เข้าถึงผู้บริโภค รวมถึงใช้กลยุทธการตัดราคาในสินค้ากลุ่มที่ price sensitive,High number of competitors in the market: ตลาดอาหารแมวในประเทศไทยเติบโตอย่างต่อเนื่อง มีคู่แข่งเข้ามาในตลาดจำนวนมากทั้งอาหารเม็ดและอาหารเปียก , Aggressive pricing strategies: คู่แข่งใช้กลยุทธ์ราคาถูกในการเข้าตลาด ได้แก่ สินค้าเป็นกลุ่มที่ดูเรื่อง Product benefits เป็นกลุ่ม Premium ตั้งราคาเท่ากลุ่ม Standard ให้ดูว่าจับต้องได้ง่าย หรือตลาดล่างที่เติบโตมาขึ้น ก็มีการเข้าตลาดด้วยการใช้กลยุทธ์ทำราคาถูกเช่นกัน,Aggressive pricing strategies: ราคาหรือโปรโมชั่นที่จำหน่ายในงานกิจกรรมต่างๆ สู้คู่แข่งขันไม่ได้,High number of competitors in the market: การแข่งขันที่รุนแรงจากจำนวนคู่แข่งที่เพิ่มขึ้นอาจส่งผลให้สูญเสียลูกค้า รายได้ และส่วนแบ่งทางการตลาด, Aggressive pricing strategies: การลดราคาหนักในตลาด รวมถึงการส่งสินค้าฟรี และมีโปรโมชั่นพิเศษอย่างต่อเนื่อง,High number of competitors in the market: ผู้ผลิตทั้ง Branded & OEM มีจำนวนมากขึ้น และมีการนำเข้าสินค้าจากต่างประเทศมากขึ้น, Aggressive pricing strategies: ผู้ผลิตรายย่อยและบริษัทที่ทำธุรกิจจ้างผลิตที่มีต้นทุนการผลิตต่ำได้เปรียบในการแข่งขันด้านราคา, Other: กำลังซื้อของผู้บริโภคลดลง และการตัดสินใจซื้อยากขึ้น,High number of competitors in the market: ธุรกิจอาหารสัตว์เลี้ยงเติบโตมากอย่างต่อเนื่องทำให้ผู้ผลิตอาหารอื่นๆเข้ามาในตลาดนี้มากขึ้น, Aggressive pricing strategies: ผู้ผลิตรายย่อยที่มีต้นทุนต่ำได้เปรียบในการแข่นขันด้านราคา'} 